In [4]:
import Pfeature
import torch
import pandas as pd
import numpy as np
import random
import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    matthews_corrcoef,
    roc_auc_score)

from pathlib import Path
from pathlib import PurePosixPath

import os
import tempfile
import pandas as pd



In [70]:
from Pfeature.pfeature import aac_wp, atc_wp, btc_wp, pcp_wp, sep_wp, ctc_wp

import pandas as pd
import tempfile
import os

def select_datasets(files):
    # Define higher, lower and test data
    lower_level = None
    higher_level = None
    target = None

    for file in files.iterdir():
        if 'Lower' in file.name:
            lower_level = file
        elif 'Higher' in file.name:
            higher_level = file
        elif 'Target' in file.name:
            target = file

    return lower_level, higher_level, target


def sliding_window(df, window_size=1024, stride=512):
    """
    Function to create 1024 length splits for proteins >1024 in length, using stride length of 512
    Inserts new splits into a datatable with split suffix
    """
    rows = []

    for _, row in df.iterrows():
        seq = row['sequence']
        seq_len = len(seq)

        if seq_len <= window_size:
            rows.append(row.copy())
            continue

        start = 0
        split = 1
        while start < seq_len:
            # End slicing variable
            end = start + window_size

            # Create copy of the row
            new_row = row.copy()
            # Slice amino acid sequence by start end index
            new_row['sequence'] = seq[start:end]

            # Slice columns containing lists
            for col in ['label', 'position']:
                new_row[col] = row[col][start:end]

            # Add split suffix to Info_protein_id
            new_row['Info_protein_id'] = str(new_row['Info_protein_id']) + '_' + str(split)
            # Append window to row list
            rows.append(new_row)

            # Stop after the final window
            if end >= seq_len:
                break

            start += stride
            split += 1

    return pd.DataFrame(rows).reset_index(drop=True)


def delete_unlabelled_rows(df):
    """ Function to delete rows with all unlabelled residues """
    return df[df['label'].apply(lambda labels: 0 in labels or 1 in labels)].reset_index(drop=True)


def preprocess_csv(csv_file, return_origin_df_len=False):
    """ Function to preprocess a csv file.
    Imports csv, refactors label column and masks NA values, aggregates into wide format
    applies sliding window and deletes unlabelled rows
    """
    preprocessed_df = pd.read_csv(csv_file)

    # Drop rows where 'Info_split' == 'NA'
    preprocessed_df = preprocessed_df[preprocessed_df['Info_split'] != 'NA']

    # Mask n/a values with -100
    preprocessed_df['Class'] = preprocessed_df['Class'].fillna(-100).astype('int32')
    preprocessed_df['Class'] = preprocessed_df['Class'].replace(-1, 0)
    preprocessed_df['Info_window'] = preprocessed_df['Info_window'].astype(str)

    # Sort values before aggregation
    preprocessed_df = preprocessed_df.sort_values(['Info_protein_id', 'Info_pos'])

    # Aggregate columns for wide format
    preprocessed_df = preprocessed_df.groupby(['Info_protein_id', 'Info_group', 'Info_split'], as_index=False).agg(
        sequence=('Info_AA', ''.join),
        Info_window_seq=('Info_window', list),
        label=('Class', list),
        position=('Info_pos', list))

    # Apply sliding window
    preprocessed_df = sliding_window(preprocessed_df)

    # Delete unlabelled rows
    preprocessed_df = delete_unlabelled_rows(preprocessed_df)

    return preprocessed_df


def pfeaturizer(df):

    protein_embeddings = []

    feature_functions = [
        ("aac", aac_wp),
        ("atc", atc_wp),
        ("btc", btc_wp),
        ("pcp", pcp_wp),
        ("sep", sep_wp),
        ("ctc", ctc_wp)
    ]

    for peptide_list in df["Info_window_seq"]:

        window_embeddings = []

        for seq in peptide_list:

            seq_features = []

            with tempfile.TemporaryDirectory() as tmpdir:

                fasta_file = os.path.join(
                    tmpdir,
                    "sequence.fasta"
                )

                with open(fasta_file, "w") as f:
                    f.write(">seq\n")
                    f.write(seq + "\n")

                for name, func in feature_functions:

                    output_file = os.path.join(
                        tmpdir,
                        f"{name}.csv"
                    )

                    func(fasta_file, output_file)

                    feature_df = pd.read_csv(output_file)

                    seq_features.extend(
                        feature_df.values.flatten()
                    )

            window_embeddings.append(seq_features)

        protein_embeddings.append(window_embeddings)

    return pd.DataFrame(
        {"pfeature_embeddings": protein_embeddings}
    )

In [85]:
files = Path('/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/pfeature_pipeline/Pfeature/Pfeature/Orthopoxvirus/')

lower_level, higher_level, target_level = select_datasets(files)

lower_level = preprocess_csv(lower_level)

lower_level_pfeature_embs  = pfeaturizer(lower_level)

# lower_level_pfeature_embs
    

In [94]:
# lower_level = lower_level.join(lower_level_pfeature_embs, how='left')

df_lower = lower_level
df_lower

,Info_protein_id,Info_group,Info_split,sequence,Info_window_seq,label,position,pfeature_embeddings
0,A0A0A7W5A6.1,30.0,split_03_20,MIILIFLIFSNIVLSIDYWVSFNKTIILDSNITNDNNDINGVSWNF...,"[MIILIFLI, MIILIFLIF, MIILIFLIFS, MIILIFLIFSN,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[0.0, 0.0, 0.0, 0.0, 12.5, 0.0, 0.0, 50.0, 0...."
1,A0A0A7W6W1.1,30.0,split_03_20,MIILIFLIFSNIVLSIDYWVSFNKTIILDSNITNDNNDINGVSWNF...,"[MIILIFLI, MIILIFLIF, MIILIFLIFS, MIILIFLIFSN,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[0.0, 0.0, 0.0, 0.0, 12.5, 0.0, 0.0, 50.0, 0...."
2,A0A2X0RVU9.1,63.0,split_03_20,MVVYDLLVSLSKESIDVLRFVEANLAAFNQQYIFFNIQRKNSITTP...,"[MVVYDLLV, MVVYDLLVS, MVVYDLLVSL, MVVYDLLVSLS,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[0.0, 0.0, 12.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0..."
3,A0A856Z1Y8.1,26.0,split_04_20,MIILIFLIFSNIVLSIDYWVSFNKTIILDSNITNDNNDINGVSWNF...,"[MIILIFLI, MIILIFLIF, MIILIFLIFS, MIILIFLIFSN,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[0.0, 0.0, 0.0, 0.0, 12.5, 0.0, 0.0, 50.0, 0...."
4,AAA42429.1,24.0,split_01_20,MEDRQALEEAGEEMGFPVVNISGGGRGRRGNYSNDGSGARELSLRV...,"[MEDRQALE, MEDRQALEE, MEDRQALEEA, MEDRQALEEAG,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[12.5, 0.0, 12.5, 25.0, 0.0, 0.0, 0.0, 0.0, 0..."
...,...,...,...,...,...,...,...,...
107,YP_009927206.1,13.0,split_05_20,MASGGAFCLIANDGKADKIILAQDLLNSRISNIKNVNKSYGKPDPE...,"[MASGGAFC, MASGGAFCL, MASGGAFCLI, MASGGAFCLIA,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[25.0, 12.5, 0.0, 0.0, 12.5, 25.0, 0.0, 0.0, ..."
108,YP_009927217.1,55.0,split_02_20,MDFILNISMKMEVIFKTDLRSSSQVVFHAGSLYNWFSVEIINSGRI...,"[MDFILNIS, MDFILNISM, MDFILNISMK, MDFILNISMKM,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[0.0, 0.0, 12.5, 0.0, 12.5, 0.0, 0.0, 25.0, 0..."
109,YP_009927231.1,71.0,split_03_20,MDTETSPLLSHNLSTREGIKQSTQGLLAHTIAKYPGTTAILLGILI...,"[MDTETSPL, MDTETSPLL, MDTETSPLLS, MDTETSPLLSH,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[0.0, 0.0, 12.5, 12.5, 0.0, 0.0, 0.0, 0.0, 0...."
110,YP_009927256.1,69.0,split_02_20,MADFNSPIQYLKEDSRDRTSIGSLEYDENADTMIPSFAAGLEEFEP...,"[MADFNSPI, MADFNSPIQ, MADFNSPIQY, MADFNSPIQYL,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[[12.5, 0.0, 12.5, 0.0, 12.5, 0.0, 0.0, 12.5, ..."


In [106]:

# ----------------- Preprocess lower level data for training classifier head  --------------------

# Create train / validation splits using the info split column
lower_train_dict = {}
lower_val_dict = {}
lower_train_all = {}

for fold in range(1, 6):
    split = f'split_{fold:02d}_20'

    lower_train_dict[fold] = (df_lower[df_lower['Info_split'] != split].reset_index(drop=True))

    lower_val_dict[fold] = (df_lower[df_lower['Info_split'] == split].reset_index(drop=True))

    # Create split of data for final training
    lower_train_all = df_lower.reset_index(drop=True)

batch_size = 16
device = torch.device("mps")


class SequenceDataset(torch.utils.data.Dataset):
    # Create custom dataset class - add to separate file and import
    def __init__(self, df):
        self.protein_id = df['Info_protein_id']
        self.sequence = df['sequence']
        self.pfeature_embeddings = df['pfeature_embeddings']
        self.position = df['position']
        self.labels = df['label']

    def __len__(self):
        return len(self.sequence)

    def __getitem__(self, idx):
        return {
            'protein_id': self.protein_id.iloc[idx],
            'sequence': self.sequence.iloc[idx],
            'pfeature_embeddings': self.pfeature_embeddings[idx],
            'position': self.position.iloc[idx],
            'label': self.labels.iloc[idx]
        }


def collate_fn(batch):
    """
    Function to create a custom collator to maintain length of items within the batch
    """
    return {
        'protein_id': [x['protein_id'] for x in batch],
        'sequence': [x['sequence'] for x in batch],
        'pfeature_embeddings': [x['pfeature_embeddings'] for x in batch],
        'position': [x['position'] for x in batch],
        'label': [x['label'] for x in batch]
    }


def batch_create(dataset, batch_size):
    """
    Function to create a DataLoader instance using the custom collate function
    """

    # Create a DataLoader instance using the cv dataset and the custom collate function
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        # num_workers=4,
        # pin_memory=True,
        # persistent_workers=True
    )

    emb_output_list = []

    # Loop through each batch of tensors:
    for batch in loader:

        # embeddings = batch['pfeature_embeddings']   #.to(device)
        # labels = batch['label']  #.to(device)

        # Convert embeddings to tensors
        embeddings = [torch.tensor(x) for x in batch['pfeature_embeddings']]
        
        # Convert labels to tensors
        labels = [torch.tensor(x) for x in batch['label']]

        emb_output_list.append({
            'embeddings': embeddings,
            'labels': labels,
        })

        # Pad each label to the size of the largest embedding within the batch
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    

    return emb_output_list


def create_datasets_for_clf(train_dict, val_dict, batch_size):
    """
    Function to create datasets and store in dictionaries for cross validation
    """

    train_datasets = {
        key: SequenceDataset(value)
        for key, value in train_dict.items()
    }

    val_datasets = {
        key: SequenceDataset(value)
        for key, value in val_dict.items()
    }

    train_loaded = {}
    for key, value in train_datasets.items():
        train_loaded[key] = batch_create(value, batch_size)

    val_loaded = {}
    for key, value in val_datasets.items():
        val_loaded[key] = batch_create(value, batch_size)

    return train_loaded, val_loaded


    
# Create dataset
lower_train_all = SequenceDataset(lower_train_all)

    
# Generate pfeature embeddings for lower level data 
train_loaded, val_loaded = create_datasets_for_clf(lower_train_dict, lower_val_dict, batch_size)

full_train_batched = batch_create(lower_train_all, batch_size)



# Determine pos / neg weighting for loss function
weight = class_weighting_for_clf(lower_level)
weight = weight.to(device)


# Create a parameter for the final hidden layer dim of the model to use as the classifier input dimension
embedding_dim = model.config.hidden_size

# Instantiate classifier
clf = PerResidueClassifier(embedding_dim)

# Move to CUDA
clf = clf.to(device)

# Instantiate weighted cross-entropy loss function, ignores positions with mask -100
loss_fcn = nn.CrossEntropyLoss(weight=weight, ignore_index=-100)




# -------- Train the classifier ----------------
clf_trained = train_classifier(embedding_dim, train_loaded, val_loaded, loss_fcn, epochs=clf_epochs)

# Calculate average validation metrics per epoch
clf_validation_avg_metrics = clf_trained.groupby(['epoch'])[[
    'train_loss',
    'train_acc',
    'val_loss',
    'val_acc',
    'val_precision',
    'val_recall',
    'val_f1',
    'val_mcc',
    'val_auc']].agg(['mean', 'std'])

# Save to csv
clf_validation_avg_metrics.to_csv(Path(f'{local_output}_clf_validation_avg_metrics.csv'))

# Validation metrics per epoch by fold
clf_validation_metrics = clf_trained[[
    'epoch',
    'fold',
    'train_loss',
    'train_acc',
    'val_loss',
    'val_acc',
    'val_precision',
    'val_recall',
    'val_f1',
    'val_mcc',
    'val_auc']]

# Save to csv
clf_validation_metrics.to_csv(Path(f'{local_output}_clf_validation_metrics_by_fold.csv'))

best_auc_epochs = clf_validation_avg_metrics['val_auc'].idxmax(axis=0)['mean']

print('clf_validation_avg_metrics', clf_validation_avg_metrics)

clf = clf = PerResidueClassifier(embedding_dim).to(device)

# Train classifier using all data for the best AUC number of epochs
final_clf_trained = train_final_classifier(clf, full_train_batched, loss_fcn, epochs=best_auc_epochs)

torch.save(
{
        'model_state_dict': final_clf_trained['model_state_dict'],
        'epochs': final_clf_trained['epochs'],
    },
    Path(f'{local_output}_final_classifier.pt'),
)

# Save results to csv
final_clf_trained['history'].to_csv(Path(f'{local_output}_clf_final_trained_metrics_by_epoch.csv'))

print('Final Trained Model Metrics', final_clf_trained)





# ------------------------------------------ Predictions on test data ------------------------------------------

# Preprocess training data
test_df = pd.read_csv(target)

# Mask n/a values with -100
test_df['Class'] = test_df['Class'].fillna(-100).astype('int32')
test_df['Class'] = test_df['Class'].replace(-1, 0)

# Sort values before aggregation
test_df = test_df.sort_values(['Info_protein_id', 'Info_pos'])

# Aggregate columns for wide format
test_df = test_df.groupby(['Info_protein_id'], as_index=False).agg(
    sequence=('Info_AA', ''.join),
    label=('Class', list),
    position=('Info_pos', list))

# Apply sliding window
test_df = sliding_window(test_df)

# Delete rows
test_df = delete_unlabelled_rows(test_df)

# Create datasets
test_datasets = SequenceDataset(test_df)

# Create batches and embeddings using relevant ESM2 model
test_batched = batch_create(test_datasets, batch_size, tokenizer, model, mode)


# ------ Evaluate using pretrained classifier --------

# Instantiate new model
clf = PerResidueClassifier(embedding_dim).to(device)

# Load trained clf model
checkpoint_data = torch.load(Path(f'{local_output}_final_classifier.pt'), weights_only=False)

# Load trained weights into clf
clf.load_state_dict(checkpoint_data['model_state_dict'])

# Put classifier into evaluation mode
clf.eval()

test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():

    for batch in test_batched:
        outputs = clf(batch['embeddings'].to(device))

        labels = batch['labels'].reshape(-1).to(device)

        # Calculate preds and probs
        preds = torch.argmax(outputs, dim=-1).reshape(-1)
        probs = torch.softmax(outputs, dim=-1)[:, :, 1].reshape(-1)

        test_preds.append(preds)
        test_labels.append(labels)
        test_probs.append(probs)

# Concat tensors
test_preds = torch.cat(test_preds)
test_labels = torch.cat(test_labels)
test_probs = torch.cat(test_probs)

mask = test_labels != -100

test_preds = test_preds[mask].cpu().numpy()
test_labels = test_labels[mask].cpu().numpy()
test_probs = test_probs[mask].cpu().numpy()

# Calculate metrics
accuracy = accuracy_score(test_labels, test_preds)
precision = precision_score(test_labels, test_preds)
recall = recall_score(test_labels, test_preds)
f1 = f1_score(test_labels, test_preds)
mcc = matthews_corrcoef(test_labels, test_preds)
auc = roc_auc_score(test_labels, test_probs)


# Create pandas dataframe
data = { 'Accuracy':accuracy, 'Precision':precision, 'Recall':recall, 'F1':f1, 'mcc':mcc, 'auc':auc}

test_results = pd.DataFrame(data, index=np.array(np.arange(1,2)))

# Save to csv
test_results.to_csv(Path(f'{local_output}_test_predictions.csv'))

print('test results', test_results)

# Free up memory
gc.collect()
torch.cuda.empty_cache()



return {
    'pathogen': pathogen,
    'checkpoint': checkpoint,
    'mode': mode
}
